# Computer Vision - Part 3

### OpenCV Hands-On Practice

Getting started with OpenCV for image processing and computer vision tasks. This notebook demonstrates basic operations such as reading, processing, and saving images using OpenCV.

OpenCV is a powerful library for computer vision tasks, and it provides a wide range of functions for image processing, feature detection, and more. In this notebook, we will cover some fundamental operations to get you started with OpenCV.

From code samples to a complete webcam-to-coordinates pipeline — putting everything together in the lab.

### Lab 1: Reading Images & Color Space Conversion

The starting point for every OpenCV pipeline: load an image and convert to the color spaces needed for downstream processing.

In [1]:
!pip install opencv-python


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [3]:
import cv2

# Read image from disk (BGR format by default)
img = cv2.imread("apple.jpg")
# Convert to Grayscale for thresholding / edge detection
gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# Convert to HSV for color-based segmentation
hsv = cv2.cvtColor(img, cv2.COLOR_BGR2HSV)

# Save the image
cv2.imwrite("apple_gray.jpg", gray)
cv2.imwrite("apple_hsv.jpg", hsv)

True

### Lab 2: Blur & Color Segmentation

Apply Gaussian blur to reduce noise, then create a binary mask for the target color using HSV thresholding.

In [6]:
import numpy as np
# Step 1: Gaussian blur to suppress noise before segmentation
blur = cv2.GaussianBlur(img, (5, 5), 0)
# Step 2: Convert blurred image to HSV
hsv = cv2.cvtColor(blur, cv2.COLOR_BGR2HSV)
# Step 3: Define color range (yellow/orange object example)
lower = np.array([20, 80, 80])
upper = np.array([40, 255, 255])
# Step 4: Create binary mask
mask = cv2.inRange(hsv, lower, upper)

# Save the image
cv2.imwrite("apple_blur.jpg", blur)
cv2.imwrite("apple_mask.jpg", mask)

True

### Lab 3: Contour Detection & Centroid Extraction

The complete pipeline — from cleaned mask to the final `(u, v)` target coordinate ready for robot control.

In [7]:
kernel = np.ones((5, 5), np.uint8)
# Clean mask with morphological ops
mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
# Find contours in cleaned mask
contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL,
cv2.CHAIN_APPROX_SIMPLE)
# Select largest contour (most likely the target object)
cnt = max(contours, key=cv2.contourArea)
# Compute image moments to get centroid
M = cv2.moments(cnt)
u = int(M["m10"] / M["m00"])
v = int(M["m01"] / M["m00"])
print(f"Target pixel: u={u}, v={v}")
# Draw centroid and bounding box
cv2.circle(img, (u, v), 5, (0, 255, 0), -1)
x, y, w, h = cv2.boundingRect(cnt)
cv2.rectangle(img, (x, y), (x+w, y+h), (255, 0, 0), 2)

Target pixel: u=769, v=513


array([[[131, 117, 119],
        [132, 118, 120],
        [131, 117, 119],
        ...,
        [ 60,  57,  59],
        [ 60,  57,  59],
        [ 62,  59,  61]],

       [[131, 117, 119],
        [131, 117, 119],
        [130, 116, 118],
        ...,
        [ 64,  57,  60],
        [ 64,  57,  60],
        [ 66,  59,  62]],

       [[132, 116, 117],
        [131, 115, 116],
        [131, 115, 116],
        ...,
        [ 71,  57,  61],
        [ 71,  57,  61],
        [ 72,  58,  62]],

       ...,

       [[212, 201, 197],
        [218, 207, 203],
        [218, 209, 205],
        ...,
        [  2,   3,   1],
        [  2,   3,   1],
        [  2,   3,   1]],

       [[218, 207, 203],
        [223, 212, 208],
        [223, 214, 210],
        ...,
        [  2,   3,   1],
        [  2,   3,   1],
        [  2,   3,   1]],

       [[222, 211, 207],
        [228, 217, 213],
        [229, 220, 216],
        ...,
        [  2,   3,   1],
        [  2,   3,   1],
        [  2,   3,   1]]